In [2]:
# https://youtu.be/tepxdcepTbY
"""
@author: Sreenivas Bhattiprolu

Code tested on Tensorflow: 2.2.0
    Keras: 2.4.3

dataset: https://finance.yahoo.com/quote/GE/history/
Also try S&P: https://finance.yahoo.com/quote/%5EGSPC/history?p=%5EGSPC
"""

import os

os.environ["KERAS_BACKEND"] = "torch"

import keras
import numpy as np
from keras.models import Sequential
from keras.layers import LSTM
from keras.layers import Dense, Dropout
import pandas as pd
from matplotlib import pyplot as plt
from sklearn.preprocessing import StandardScaler
from pathlib import Path
import seaborn as sns
from natsort import natsorted
from math import floor, log10
from sklearn.model_selection import train_test_split
import shutil

In [3]:
def merge_datasets(datasets):
    """
    Merges a list of datasets (NumPy arrays or Pandas DataFrames) into a single dataset.

    Args:
        datasets: A list of datasets, where each dataset is a NumPy array or a Pandas DataFrame.
                  It is assumed that all datasets have the same number of columns (features).
                  If the datasets have different numbers of columns, the function will still
                  execute, but the results may not be meaningful for subsequent processing.
                  The function does not check for this.

    Returns:
        A single Pandas DataFrame containing all the data from the input datasets.
        Returns an empty DataFrame if the input list is empty.
    """
    if not datasets:
        return pd.DataFrame()  # Return an empty DataFrame if the list is empty

    # Use pd.concat for efficient merging of DataFrames
    merged_data = pd.concat(datasets, axis=0, ignore_index=True)
    return merged_data

def preprocess_data(data, Tr_data, scaler=None):
    """
    Preprocesses the input data by scaling it using StandardScaler.

    Args:
        data: A Pandas DataFrame representing the data to be preprocessed.
        scaler: (Optional) A pre-initialized StandardScaler object. If None,
            a new StandardScaler is initialized and used. If provided,
            the provided scaler is used to transform the data. This is
            useful for applying the same scaling to multiple datasets.
        Tr_data
    Returns:
        A tuple containing:
        - The scaled data as a Pandas DataFrame.
        - The StandardScaler object used for scaling. If a scaler was
          passed in, the same scaler is returned.
    """
    # Ensure we are working with a DataFrame
    
    data = pd.DataFrame(data)
    Tr_data = pd.DataFrame(Tr_data)
    
    if scaler is None:
        scaler = StandardScaler()
        # Fit and transform, then convert back to DataFrame
        scaled_data = scaler.fit_transform(data)
        scaled_data = pd.DataFrame(scaled_data, columns=data.columns)
        # Keep column names
        print(data.columns)
        
        scaled_Tr_data = scaler.transform(Tr_data)
        scaled_Tr_data = pd.DataFrame(scaled_Tr_data, columns=data.columns)
        
    else:
        # Transform using the provided scaler
        scaled_data = scaler.transform(data)
        scaled_data = pd.DataFrame(scaled_data, columns=data.columns)
    return scaled_data, scaled_Tr_data, scaler

def unmerge_datasets(merged_data, original_shapes):
    """
    Unmerges a single dataset into a list of datasets, given the original shapes.

    Args:
        merged_data: A Pandas DataFrame representing the merged dataset.
        original_shapes: A list of tuples, where each tuple represents the shape
            (number of rows, number of columns) of the original dataset.
            The number of columns is not strictly required, but is good practice
            to include. Only the number of rows is used to split the data.
            If the shapes do not add up to the total number of rows in
            `merged_data`, the function will raise a ValueError.

    Returns:
        A list of Pandas DataFrames, where each DataFrame represents an unmerged dataset.
        Returns an empty list if `merged_data` or `original_shapes` is empty.

    Raises:
        ValueError: If the total number of rows in the original shapes do not
            match the number of rows in the merged data.
    """
    if merged_data.empty or not original_shapes:
        return []

    total_rows = sum(rows for rows, _ in original_shapes)
    if total_rows != merged_data.shape[0]:
        raise ValueError(
            f"Total rows in original shapes ({total_rows}) do not match the number of rows in the merged data ({merged_data.shape[0]})"
        )

    unmerged_datasets = []
    start_index = 0
    for rows, _ in original_shapes:
        end_index = start_index + rows
        unmerged_data = merged_data.iloc[start_index:end_index].copy() # Use .iloc for DataFrame slicing and .copy()
        unmerged_datasets.append(unmerged_data)
        start_index = end_index
    return unmerged_datasets


def read_datasets_from_directory(directory, targets, record_shapes=False):
    """
    Reads all CSV files from a specified directory into a list of Pandas DataFrames.

    Args:
        directory: The path to the directory containing the CSV files.
        record_shapes: Boolean indicating whether to record the shapes of the
                       read datasets.  If True, the function also returns a list
                       of the shapes.

    Returns:
        A tuple containing:
        - A list of Pandas DataFrames, where each DataFrame represents a dataset
          read from a CSV file.
        - A list of tuples, where each tuple represents the shape
          (number of rows, number of columns) of the corresponding dataset.
          This is only returned if record_shapes is True.
          Returns an empty list if the directory is empty or does not exist,
          or if no CSV files are found.  Raises an error if a non-CSV file
          is encountered.
    """
    datasets = []
    original_shapes = []
    try:
        # Use os.listdir to get all files in the directory
        for filename in os.listdir(directory):
            redex = targets
            if filename.endswith(".csv"):
                filepath = os.path.join(directory, filename)
                try:
                    # Read the CSV file into a Pandas DataFrame
                    df = pd.read_csv(filepath).iloc[:, 1:]
                    
                    # for j in list(df):
                    #     if j not in redex:
                    #         redex.append(j)
                    #     else:
                    #         continue
                    # 
                    df = df.reindex(
                        columns=redex)
                    
                    if record_shapes:
                        original_shapes.append(df.shape)  # Record shape before appending
                    datasets.append(df)
                except Exception as e:
                    print(f"Error reading file {filename}: {e}") # Inform about file reading errors
            elif os.path.isfile(os.path.join(directory, filename)):
                print(f"Skipping non-CSV file: {filename}") # Inform about skipping non-CSV
        if not datasets:
            print(f"No CSV files found in directory: {directory}")
    except FileNotFoundError:
        print(f"Directory not found: {directory}")
        return [], []  # Return empty lists if directory not found
    except Exception as e:
        print(f"An error occurred: {e}")
        return [], [] # Return empty list if other error.

    if record_shapes:
        return datasets, original_shapes
    else:
        return datasets, []

def data_prep(Dataset_Dir, test_dataset_index, targets):
    loaded_datasets, original_shapes = read_datasets_from_directory((Dataset_Dir), targets, record_shapes=True)
    if not loaded_datasets:
        print("No datasets loaded. Exiting.")

    test_dataset = loaded_datasets.pop(test_dataset_index)  # Remove the test dataset
    original_shapes.pop(test_dataset_index) # Remove the corresponding shape


    print("Original dataset shapes:")
    for shape in original_shapes:
        # print(shape)
        continue
    # Merge the datasets
    merged_data = merge_datasets(loaded_datasets)
    # print("\nShape of merged data:", merged_data.shape)

    # Preprocess the merged data
    scaled_data, scaled_test_data, scaler = preprocess_data(merged_data, test_dataset)
    # print("\nShape of scaled data:", scaled_data.shape)

    # Unmerge the data
    unmerged_datasets = unmerge_datasets(scaled_data, original_shapes)
    print(f"\nUnmerged dataset shape:")
    for shape in [dataset.shape for dataset in unmerged_datasets]:
        # print(shape)
        continue 
    
    return unmerged_datasets, scaled_test_data, scaler

In [32]:
Dataset_Dir =  Path(r'C:\Users\2MY\Documents\Uni Work\Level 4\TR-Y4-Project\LSTM_Prac\ML_Data')
test_cell_indexes = [7,11,13]

target_var_1 = 'xcentre'
target_var_2 = 'ycentre'
target_list = [target_var_1, target_var_2]


UniDim_results_Data_Dir = Path.joinpath(Path(os.getcwd()),'UniDim_results_Data')

# if UniDim_results_Data_Dir.is_dir():
#     shutil.rmtree(UniDim_results_Data_Dir)
# UniDim_results_Data_Dir.mkdir(exist_ok=True)

#T_list = [t2]
target_num = 2
for test_num, test_ind in enumerate(test_cell_indexes):
    test_results_Dir = Path.joinpath(UniDim_results_Data_Dir, f'Cell_test_{test_num}')
    if test_results_Dir.is_dir():
        shutil.rmtree(test_results_Dir)
    test_results_Dir.mkdir(exist_ok=True)
    
    for win in range(2,3):
        n_future = 1  # Number of days we want to look into the future based on the past days.
        n_past = win
        
        N = n_future + n_past
        scaled_train_dataset, scaled_test_dataset, scaler = data_prep(Dataset_Dir, test_ind, target_list)
        testX=[]
        testY=[]
        for i in range(n_past, len(scaled_test_dataset) - n_future):
            testX.append(scaled_test_dataset.iloc[i - n_past:i, 0:2])
            testY.append(scaled_test_dataset.iloc[i + n_future - 1:i + n_future, 0:target_num]) #encodes number of targets in Y dataset
        
        null_X_train, X_test, null_y_train, y_test = train_test_split(testX, testY, test_size =1, random_state=16)
        trainX = []
        trainY = []
        
        #Read the csv file
        for i, cell in enumerate(scaled_train_dataset):
            if i < 100 :
                 
                if cell.shape[0] > N:
        
                    #df_for_training_scaled = df[cols].astype(float).round(decimals=2)
                    
                    for i in range(n_past, len(cell) - n_future):
                        trainX.append(cell.iloc[i - n_past:i, 0:cell.shape[1]])
                        trainY.append(cell.iloc[i + n_future - 1:i + n_future, 0:cell.shape[1]]) #encodes number of targets in Y dataset
                                
                else:
                    continue
           
        X_train, null_X_test, y_train, null_y_test = train_test_split(trainX, trainY, test_size =1, random_state=16)
        
        X_train, X_test, y_train, y_test = np.array(X_train), np.array(X_test), np.array(y_train), np.array(y_test)
        
        
        print('X_train shape == {}.'.format(X_train.shape))
        print('y_train shape == {}.'.format(y_train.shape))
        
        print('X_test shape == {}.'.format(X_test.shape))
        print('y_test shape == {}.'.format(y_test.shape))
        acc_df = []
        
        
        for epos in range(1,9):
            
            x_model = Sequential()
            x_model.add(keras.Input(shape=(X_train.shape[1], X_train.shape[2])))
            x_model.add(LSTM(150, activation='relu', return_sequences=True))
            x_model.add(LSTM(64, activation='relu'))
            x_model.add(Dense(64))
            x_model.add(Dropout(0.2))
            x_model.add(Dense(1))
            x_model.compile(optimizer='adam', loss='mse')
        
            y_model = Sequential()
            y_model.add( keras.Input(shape=(X_train.shape[1], X_train.shape[2])))
            y_model.add(LSTM(150, activation='relu', return_sequences=True))
            y_model.add(LSTM(64, activation='relu'))
            y_model.add(Dense(64))
            y_model.add(Dropout(0.2))
            y_model.add(Dense(1))
            y_model.compile(optimizer='adam', loss='mse')
        
            x_model.fit(X_train, y_train[:,:,0], epochs=epos, batch_size=16, validation_split=0.2, verbose=1)
            y_model.fit(X_train, y_train[:,:,1], epochs=epos, batch_size=16, validation_split=0.2, verbose=1)
            
            x_stack = np.vstack((x_model.predict(X_test), y_test[:,:,0], X_test[0,-1,0]))
            y_stack = np.vstack((y_model.predict(X_test), y_test[:,:,1], X_test[0,-1,1]))
            
            prediction_copies = np.hstack((x_stack, y_stack, np.zeros((x_stack.shape[0],cell.shape[1] - y_test.shape[2]))))
            
            acc_df.append(scaler.inverse_transform(prediction_copies)[:, :2])
            
        results = np.array(acc_df).reshape(len(acc_df), np.shape(acc_df)[1]*2)
        results = pd.DataFrame(np.c_[np.arange(1,len(acc_df) + 1), results], columns=['epoch',  'x_pred', 'y_pred','x_truth','y_truth','x_truth-1','y_truth-1'])
        
        results['x_acc'] = (abs(results['x_pred'] - results['x_truth'])/results['x_truth']) * 100
        results['y_acc'] = (abs(results['y_pred'] - results['y_truth'])/results['y_truth']) * 100
        
        
        # filepath = Path.joinpath(test_results_Dir, f'Win_range_{win}.csv')
        # results.to_csv(filepath, index=False)

Original dataset shapes:
Index(['xcentre', 'ycentre'], dtype='object')

Unmerged dataset shape:
X_train shape == (874, 2, 2).
y_train shape == (874, 1, 2).
X_test shape == (1, 2, 2).
y_test shape == (1, 1, 2).
44/44 ━━━━━━━━━━━━━━━━━━━━ 1s 25ms/step - loss: 0.7206 - val_loss: 0.0294
44/44 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - loss: 0.7947 - val_loss: 0.0306
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step
Epoch 1/2
44/44 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - loss: 0.7674 - val_loss: 0.0237
Epoch 2/2
44/44 ━━━━━━━━━━━━━━━━━━━━ 1s 26ms/step - loss: 0.0224 - val_loss: 0.0042
Epoch 1/2
44/44 ━━━━━━━━━━━━━━━━━━━━ 1s 26ms/step - loss: 0.7675 - val_loss: 0.0227
Epoch 2/2
44/44 ━━━━━━━━━━━━━━━━━━━━ 1s 26ms/step - loss: 0.0197 - val_loss: 0.0030
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
Epoch 1/3
44/44 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - loss: 0.7252 - val_loss: 0.0211
Epoch 2/3
44/44 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - loss: 0.0251 - val

In [33]:
results

,epoch,x_pred,y_pred,x_truth,y_truth,x_truth-1,y_truth-1,x_acc,y_acc
0,1.0,566.198844,447.909223,604.858215,463.124634,606.884338,466.96817,6.391477,3.285381
1,2.0,614.288586,460.217261,604.858215,463.124634,606.884338,466.96817,1.559104,0.627773
2,3.0,612.896815,451.352559,604.858215,463.124634,606.884338,466.96817,1.329006,2.541881
3,4.0,617.726292,449.023667,604.858215,463.124634,606.884338,466.96817,2.127453,3.044746
4,5.0,612.429053,459.774187,604.858215,463.124634,606.884338,466.96817,1.251672,0.723444
5,6.0,621.183441,463.616939,604.858215,463.124634,606.884338,466.96817,2.699017,0.106301
6,7.0,611.755996,467.462160,604.858215,463.124634,606.884338,466.96817,1.140396,0.936579
7,8.0,616.678935,456.011731,604.858215,463.124634,606.884338,466.96817,1.954296,1.535851


In [29]:
x_stack.shape[0]

1

In [28]:
np.zeros((x_stack.shape[1],cell.shape[1] - y_test.shape[2]))

array([], shape=(1, 0), dtype=float64)

In [22]:
X_test[0,-1,1]

-0.5451438661743874

In [21]:
plt.plot(x_model.history['loss'], label='Training loss')
plt.plot(x_model.history['val_loss'], label='Validation loss')
plt.legend()

TypeError: 'History' object is not subscriptable